# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [2]:
df = pd.read_csv("C:/Users/mjlia/perspectiva-dado/Scraping/dados_olimpiadas_exame.csv")
print(len(df))
print(df)

333
                                                   url  \
0    https://exame.com/esporte/tati-weston-webb-ven...   
1    https://exame.com/esporte/olimpiadas-de-invern...   
2    https://exame.com/esporte/paris-2024-futebol-o...   
3    https://exame.com/esporte/apos-bater-em-duas-b...   
4    https://exame.com/carreira/da-roca-para-paris-...   
..                                                 ...   
328  https://exame.com/esporte/nas-filipinas-ginast...   
329  https://exame.com/esporte/que-dia-acaba-a-olim...   
330  https://exame.com/esporte/historiadores-tentam...   
331  https://exame.com/esporte/olimpiadas-de-paris-...   
332  https://exame.com/esporte/quem-e-pat-burgener-...   

                                                titulo  \
0    Tati Weston-Webb vence número um do mundo no s...   
1    Olimpíadas de Inverno: hoje tem Brasil no Bobs...   
2    Paris 2024: Futebol olímpico terminará com a f...   
3    Após bater em duas barreiras, Alison dos Santo...   
4    Da r

In [1]:
import json
from pathlib import Path

import pandas as pd

In [3]:
df = pd.read_csv("C:/Users/mjlia/perspectiva-dado/Scraping/dados_olimpiadas_exame.csv")
print(len(df))

333


In [8]:
noticias = []

#for arquivo in sorted(Path("data").glob("*.json")):
#    with arquivo.open(encoding="utf-8") as f:
#        noticias.append(json.load(f))

df = pd.DataFrame(noticias)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 0 noticias
Depois: 0 noticias


""


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [4]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,Surfe feminino: veja quem se classificou para ...,surfe feminino veja quem se classificou para a...
1,Edson Luques Bindilatti e Luis Bacca Gonçalves...,edson luques bindilatti e luis bacca goncalves...
2,Calendários das partidas das competições foram...,calendarios das partidas das competicoes foram...
3,Alison dos Santos: atleta é medalhista olímpic...,alison dos santos atleta e medalhista olimpico...
4,"Maranhão, atleta olímpico de arremedo de peso:...",maranhao atleta olimpico de arremedo de peso a...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [5]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,surfe feminino veja quem se classificou para a...,"[surfe, feminino, veja, classificou, quartas, ...",surfe feminino veja classificou quartas final ...
1,edson luques bindilatti e luis bacca goncalves...,"[edson, luques, bindilatti, luis, bacca, gonca...",edson luques bindilatti luis bacca goncalves v...
2,calendarios das partidas das competicoes foram...,"[calendarios, partidas, competicoes, confirmad...",calendarios partidas competicoes confirmados j...
3,alison dos santos atleta e medalhista olimpico...,"[alison, santos, atleta, medalhista, olimpico,...",alison santos atleta medalhista olimpico campe...
4,maranhao atleta olimpico de arremedo de peso a...,"[maranhao, atleta, olimpico, arremedo, peso, 1...",maranhao atleta olimpico arremedo peso 13 anos...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,00024,0009,000m,003,0041,01,02,023,...,zona,zonas,zone,zones,zoologicos,zotto,zov6pfqais,zozelw8n8a,zuerlein,zvi
0,1,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [7]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

1403 colunas removidas


,aab,aabertura,aadk,aalemanhacompleta,aangolagarantiu,aansiedade,aanterselva,aantiguidade,aarena,aargentina,...,zjfesjcqxs,zoagaharlley,zona,zonas,zone,zones,zoologicos,zotto,zuerlein,zvi
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
df_bow.sum()

aab                  1
aabertura            2
aadk                 1
aalemanhacompleta    1
aangolagarantiu      1
                    ..
zones                4
zoologicos           1
zotto                2
zuerlein             1
zvi                  1
Length: 17199, dtype: int64

### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [8]:
metadados = df[["data", "tags", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,tags,titulo,url,bow_aab,bow_aabertura,bow_aadk,bow_aalemanhacompleta,bow_aangolagarantiu,bow_aansiedade,...,bow_zjfesjcqxs,bow_zoagaharlley,bow_zona,bow_zonas,bow_zone,bow_zones,bow_zoologicos,bow_zotto,bow_zuerlein,bow_zvi
0,Sem data,NaN,Tati Weston-Webb vence número um do mundo no s...,https://exame.com/esporte/tati-weston-webb-ven...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Sem data,NaN,Olimpíadas de Inverno: hoje tem Brasil no Bobs...,https://exame.com/esporte/olimpiadas-de-invern...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Sem data,NaN,Paris 2024: Futebol olímpico terminará com a f...,https://exame.com/esporte/paris-2024-futebol-o...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Sem data,NaN,"Após bater em duas barreiras, Alison dos Santo...",https://exame.com/esporte/apos-bater-em-duas-b...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Sem data,NaN,Da roça para Paris: a história inspiradora do ...,https://exame.com/carreira/da-roca-para-paris-...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [9]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 17199


bow_paris         1625
bow_jogos         1563
bow_brasil        1086
bow_olimpicos     1027
bow_exame          812
bow_atletas        741
bow_whatsapp       669
bow_domingo        642
bow_olimpiadas     595
bow_medalhas       572
dtype: int64

In [10]:
frequencia_palavras.tail(10)

bow_zirong              1
bow_zixu                1
bow_zixuultrapassou     1
bow_zjfesjcqxs          1
bow_zoagaharlley        1
bow_zimbabuana          1
bow_zimbabuanakirsty    1
bow_zimbabue            1
bow_zimbabuense         1
bow_aatleta             1
dtype: int64

In [11]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

noticias     333
whatsapp     333
publicado    333
receba       333
exame        333
mail         319
invalido     315
brasil       311
jogos        307
mundo        288
dtype: int64

In [12]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
78,"Jogos de hoje, sexta-feira, 2; onde assistir e...",82
154,Quantos dias faltam para as Olimpíadas 2024?,105
9,"Argentina x Uruguai: onde assistir, horários e...",109
257,Lucas Pinheiro Braathen perde medalha no slalom,112
60,Conselho de Ética do COB anuncia mudança na pr...,121
198,"Brasil x Colômbia: onde assistir, horários e e...",123
100,"Canadá x Brasil: onde assistir, horário e esca...",124
127,Guilherme Toldo representará o Brasil na esgri...,126
276,Seleção masculina de handebol fica fora dos Jo...,126
250,Confira o calendário completo das Olimpíadas d...,127
